# Первичное знакомство с датасетом Northwind

Цель этого ноутбука — первый осмотр данных: понять, что вообще лежит в `data/raw/`, прежде чем делать любой анализ.

**Что мы делаем:**
- загружаем все CSV-файлы;
- смотрим на размер, названия столбцов, первые строки и количество пропусков в каждой таблице.

**Что мы НЕ делаем в этом ноутбуке:**
- не строим графики;
- не делаем выводов о бизнесе (например, «какие товары продаются лучше»);
- не проверяем ключи и связи между таблицами — это отдельный, следующий шаг.

Описание смысла таблиц и столбцов — в [`data/data_dictionary.md`](../data/data_dictionary.md).

In [ ]:
# Библиотека для работы с табличными данными
import pandas as pd

# Path удобнее строк при работе с путями к файлам
from pathlib import Path

# Папка с исходными (сырыми) CSV-файлами — их нельзя менять вручную
RAW_DIR = Path("../data/raw")

## Вспомогательная функция для осмотра таблицы

Дальше нам нужно повторить один и тот же набор проверок для каждой из 9 таблиц. Чтобы не копировать одинаковый код 9 раз, оформим его один раз в виде функции `explore_table`.

In [ ]:
def explore_table(df, name):
    """
    Печатает базовую информацию о таблице:
    - количество строк и столбцов;
    - названия столбцов;
    - первые 5 строк;
    - количество пропущенных значений по каждому столбцу.
    """
    print(f"=== {name} ===")

    # df.shape возвращает кортеж (количество строк, количество столбцов)
    n_rows, n_cols = df.shape
    print(f"Количество строк: {n_rows}")
    print(f"Количество столбцов: {n_cols}")

    # Список названий столбцов таблицы
    print("\nСтолбцы:")
    print(list(df.columns))

    # Первые 5 строк — чтобы визуально увидеть, как выглядят данные
    print("\nПервые 5 строк:")
    display(df.head())

    # Считаем пропуски по каждому столбцу.
    # По умолчанию pandas распознаёт строку "NULL" как пропуск (NaN),
    # поэтому дополнительная настройка здесь не нужна.
    print("\nПропущенные значения по столбцам:")
    print(df.isnull().sum())

    print("\n")

## Особенность трёх файлов: customers, suppliers, orders

При прямой загрузке `customers.csv`, `suppliers.csv` и `orders.csv` через `pd.read_csv` возникает ошибка парсинга. Причина: в адресных полях этих файлов встречаются значения с запятой внутри (например, `24, place Kléber`), но сам исходный CSV не берёт такие значения в кавычки, как того требует формат CSV. Из-за этого адрес распадается на два отдельных поля, и вся строка "съезжает".

Это подтверждённая особенность исходных файлов (не наша догадка и не проблема кодировки — кодировка файлов корректная, UTF-8). Чтобы не терять такие строки и не искажать данные, ниже — функция, которая читает файл построчно и склеивает обратно "разорванный" адрес по номеру столбца.

In [ ]:
import csv

def read_csv_fix_address_commas(path, address_col_index):
    """
    Читает CSV, в котором адресное поле иногда содержит запятую без кавычек
    (например, "24, place Kléber"), из-за чего строка распадается на лишнее поле.
    Склеивает такие "разорванные" поля обратно в одно значение адреса.

    path - путь к CSV-файлу
    address_col_index - индекс столбца с адресом (считая с 0, по заголовку файла)
    """
    with open(path, encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        n_expected = len(header)

        rows = []
        for row in reader:
            # Пока в строке больше полей, чем должно быть по заголовку,
            # склеиваем поле адреса со следующим за ним полем
            while len(row) > n_expected:
                row[address_col_index] = row[address_col_index] + "," + row[address_col_index + 1]
                del row[address_col_index + 1]
            rows.append(row)

    return pd.DataFrame(rows, columns=header)

## Categories — категории товаров

Справочник категорий товаров. Проверяем, сколько категорий всего и нет ли пропусков в названии или описании.

In [ ]:
df_categories = pd.read_csv(RAW_DIR / "categories.csv")
explore_table(df_categories, "categories")

## Customers — клиенты

Справочник клиентов компании. Проверяем полноту контактных данных: адрес, город, регион, телефон и т.д.

In [ ]:
df_customers = read_csv_fix_address_commas(RAW_DIR / "customers.csv", address_col_index=4)  # address
explore_table(df_customers, "customers")

## Employees — сотрудники

Данные о сотрудниках компании. В таблице есть технические столбцы `photo` и `photoPath` (бинарные данные и путь к фото) — они попадут в осмотр наравне с остальными, но в дальнейшем анализе использоваться не будут (см. `data_dictionary.md`).

In [ ]:
df_employees = pd.read_csv(RAW_DIR / "employees.csv")
explore_table(df_employees, "employees")

## Employee-Territories — связь сотрудников и территорий

Служебная таблица-связка между сотрудниками и территориями продаж. Ожидаем всего два столбца-идентификатора без содержательных атрибутов.

In [ ]:
df_employee_territories = pd.read_csv(RAW_DIR / "employee-territories.csv")
explore_table(df_employee_territories, "employee-territories")

## Territories — территории продаж

Справочник территорий продаж.

In [ ]:
df_territories = pd.read_csv(RAW_DIR / "territories.csv")
explore_table(df_territories, "territories")

## Suppliers — поставщики

Справочник поставщиков товаров.

In [ ]:
df_suppliers = read_csv_fix_address_commas(RAW_DIR / "suppliers.csv", address_col_index=4)  # address
explore_table(df_suppliers, "suppliers")

## Products — товары

Справочник товаров: цена, остатки на складе, привязка к категории и поставщику.

In [ ]:
df_products = pd.read_csv(RAW_DIR / "products.csv")
explore_table(df_products, "products")

## Orders — заказы

Шапка заказа: кто заказал, кто оформил, даты и адрес доставки. Проверяем, есть ли пропуски в датах и адресных полях.

In [ ]:
df_orders = read_csv_fix_address_commas(RAW_DIR / "orders.csv", address_col_index=9)  # shipAddress
explore_table(df_orders, "orders")

## Order Details — позиции заказа

Самая крупная таблица: конкретные товары, цены и количества внутри каждого заказа.

In [ ]:
df_order_details = pd.read_csv(RAW_DIR / "order-details.csv")
explore_table(df_order_details, "order-details")

## Итог первого осмотра

Мы загрузили все 9 таблиц Northwind и посмотрели на их размер, структуру столбцов, первые строки и количество пропусков. Никаких выводов о бизнесе и графиков в этом ноутбуке не делалось — это была только первичная разведка данных.

**Следующий шаг:** детальная валидация данных (проверка ключей, связей между таблицами, качества значений) — см. `VALIDATION_CHECKLIST.md` и план в `README.md`.